In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

In [2]:
import assignment2_tools


In [3]:
from openai import OpenAI
import json

client = OpenAI()

tools = [
    {
        "type": "function",
        "name": "fetch_on_air_movie",
        "description": "Get a list of movies that is on air",
        "parameters": {
            "type": "object",
            "properties": {
                "max_results": {
                        "type": "integer",
                        "description": "Maximum number of movies to return",
                    },
            },
             "required": ["max_results"],
            "additionalProperties": False,
         },
        "strict": True,
    },
]

In [4]:
input_list = [
    # {"role": "user", "content": "How many movies that is on air?"},
     {"role": "user", "content": "Is Blue Signal on air right now? Suggest me some similar movies also on air"}
]

In [5]:
response = client.responses.create(
    model="gpt-5",
    tools=tools,
    input=input_list,
)

In [6]:
response.output

[ResponseReasoningItem(id='rs_0f0e45783b06ff2b0069117c005d30819cbff0073fca064253', summary=[], type='reasoning', content=None, encrypted_content=None, status=None),
 ResponseFunctionToolCall(arguments='{"max_results": 200}', call_id='call_rVM7fvx9gvCpoUQ0V2hpB0gt', name='fetch_on_air_movie', type='function_call', id='fc_0f0e45783b06ff2b0069117c10ad1c819c935721a20069b0e9', status='completed')]

In [7]:
print(response.output[1].to_json())

{
  "arguments": "{\"max_results\": 200}",
  "call_id": "call_rVM7fvx9gvCpoUQ0V2hpB0gt",
  "name": "fetch_on_air_movie",
  "type": "function_call",
  "id": "fc_0f0e45783b06ff2b0069117c10ad1c819c935721a20069b0e9",
  "status": "completed"
}


In [8]:
input_list += response.output

for item in response.output:
    if item.type == "function_call":
        if item.name == "fetch_on_air_movie":
            # # Execute the function logic for fetc
            # horoscope = fetch_on_air_movie(**json.loads(item.arguments))
            
            # Provide function call results to the model
            input_list.append({
                "type": "function_call_output",
                "call_id": item.call_id,
                # "output": json.dumps({
                #   "horoscope": horoscope
                # })
            })
            

In [9]:
print(input_list)

[{'role': 'user', 'content': 'Is Blue Signal on air right now? Suggest me some similar movies also on air'}, ResponseReasoningItem(id='rs_0f0e45783b06ff2b0069117c005d30819cbff0073fca064253', summary=[], type='reasoning', content=None, encrypted_content=None, status=None), ResponseFunctionToolCall(arguments='{"max_results": 200}', call_id='call_rVM7fvx9gvCpoUQ0V2hpB0gt', name='fetch_on_air_movie', type='function_call', id='fc_0f0e45783b06ff2b0069117c10ad1c819c935721a20069b0e9', status='completed'), {'type': 'function_call_output', 'call_id': 'call_rVM7fvx9gvCpoUQ0V2hpB0gt'}]


In [10]:
response = client.responses.create(
    model="gpt-5",
    instructions="Only respond with the movie you found in fetch_on_air_movie.",
    tools=tools,
    input=input_list,
)


BadRequestError: Error code: 400 - {'error': {'message': "Missing required parameter: 'input[3].output'.", 'type': 'invalid_request_error', 'param': 'input[3].output', 'code': 'missing_required_parameter'}}

In [ ]:
input_list

[{'role': 'user', 'content': 'How many movies that is on air?'},
 {'role': 'user', 'content': 'Is Blue Signal on air right now?'},
 ResponseReasoningItem(id='rs_0fa60df524bd2b680069117b79725c81a18b9c5be86ea00ba4', summary=[], type='reasoning', content=None, encrypted_content=None, status=None),
 ResponseFunctionToolCall(arguments='{"max_results": 1000}', call_id='call_1CeAPRskUGJwqkpA7d9SDy4t', name='fetch_on_air_movie', type='function_call', id='fc_0fa60df524bd2b680069117b7f644081a1896919f74a153620', status='completed'),
 {'type': 'function_call_output', 'call_id': 'call_1CeAPRskUGJwqkpA7d9SDy4t'}]

In [ ]:
#print(response.model_dump_json(indent=2))
print("\n" + response.output_text)